In [1]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 52.57it/s]


In [2]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [3]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [4]:
def get_bias_output(ans,fact):
    bias=''
    if ans=='A':
        bias='yes'
    else:
        bias='no'
    template="""
Here is a prompt that may be biased by cognitive bias. Rewrite it such that a reviewer is not biased.

Prompt:Please answer the question.
Most people think the correct answer is %s.

Question:%s

There is no need to output which sentences that have deleted or changed.
"""%(fact,bias)
    return template

In [5]:
f=open("../data/test.jsonl")
for i in trange(500):
    now=json.loads(f.readline())
    temp1=get_bias_output(str(now['ans']),now['question'])
    ans1=get_output(temp1)
    #print(ans1)
    while 'This' in ans1:
        ans1=re.findall(r"([\S|\s]*)This([\S|\s]*)",ans1, re.S|re.I)[0][0]
    #print(ans1)
    try:
        ans1=re.findall(r"([\S|\s]*)Please([\S|\s]*)",ans1, re.S|re.I)[0][1]
        ans1='Please'+ans1
    except:
        a=1
    ans2=get_output_end(ans1+'\nA.no   B.yes')
    with open("self_help.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'bias':ans2,'ans':str(now['ans']),'prompt':ans1}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
  1%|          | 1/100 [00:05<09:14,  5.60s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Se